# Lecture 5: Belief Propagation

* Factor graphs
* Bethe free energy
* Belief Propagation (BP) on Markov Random Fields and Factor Graphs
* Some applications of BP

McKay, **Chapter 26**.

*Recommended reading:*
* [Understanding belief propagation and its generalizations](https://www.cs.huji.ac.il/course/2005/pmai/tirguls/TR2001-22.pdf) and [Bethe free energy, Kikuchi approximations, and belief propagation algorithms](https://www2.stat.duke.edu/~scs/Courses/Stat376/Papers/GraphicalModels/BetheKikuchiBelProp.pdf) cover most of the material in this lecture
* [Factor graphs and the sum-product algorithm](https://ieeexplore.ieee.org/document/910572) introduced *factor graphs* and the *sum-product* algorithm
* [Andrea Montanari](https://web.stanford.edu/~montanar/) and [Marc Mezard](https://en.wikipedia.org/wiki/Marc_M%C3%A9zard) wrote the wonderful book [Information, Physics, and Computation](https://web.stanford.edu/~montanar/RESEARCH/book.html). A draft is available on Montanari's website.
* [Gibbs states and the set of solutions of random constraint satisfaction problems](https://www.pnas.org/doi/abs/10.1073/pnas.0703685104) gives an account of the use of (refined) BP methods for the study of the geometry of solutions in random constrained satisfaction problems

## The problem of marginalization

Consider a distribution than can be expressed as a product of *local factors*:
$$p(x_1,x_2,x_3,x_4,x_5)=\frac{1}{Z}\psi_a(x_1,x_2)\psi_b(x_2,x_3)\psi_c(x_3,x_4)\psi_d(x_3,x_5)$$

In this examples, the functions $\psi$ only depend on two variables, but more general dependences can be considered.

We wish to compute *marginals*, e.g. $p(x_3)$:
$$p(x_3)=\sum_{x_1,x_2,x_4,x_5} p(x_1,x_2,x_3,x_4,x_5)$$

Such computation is in general exponential in the number of variables, but if we exploit the factorization structure of $p$ we can use the distributivity property to perform *local sums* over the $\psi$. In this example, we will have:

$$p(x_3) \propto \underbrace{\sum_{x_2} \underbrace{\sum_{x_1} \psi_a(x_1,x_2)}_{m_{1 2}(x_2)}\psi_b(x_2,x_3)}_{m_{2 3}(x_3)}\underbrace{\sum_{x_4} \psi_c(x_3,x_4)}_{m_{4 3}(x_3)}\underbrace{\sum_{x_5} \psi_d(x_3,x_5)}_{m_{5 3}(x_3)} =m_{2 3}(x_3)m_{4 3}(x_3)m_{5 3}(x_3)$$

where the subscripts in the functions $m$ have been chosen to map this simple example to the general approach of **Belief Propagation**.

In order to formulate this idea in the most general case, it is good to devise some graphical depiction of the factorization/independence property of a distribution $p$.

## Three examples of factorized probability distributions

#### Factor graph

<center><img src="figs/factor_graph_kschischang.png" width=300></center>

The joint probability distribution is represented with a **bipartite graph** where local (compatibility) functions are represented as *factor nodes*:

$$p\left(\left\{ x\right\} \right)=\frac{1}{Z}\prod_{a}\psi_{a}\left(\left\{ x\right\} _{a}\right)$$

For the specific example in the figure we have (calling the functions $f$ as in the figure):

$$p\left(\left\{ x\right\} \right)=f_{A}\left(x_{1}\right)f_{B}\left(x_{2}\right)f_{C}\left(x_{1},x_{2},x_{3}\right)f_{D}\left(x_{3},x_{4}\right)f_{E}\left(x_{3},x_{5}\right)$$

#### Bayesian network

<center><img src="figs/bayesian_network_yedidia.png" width=300></center>

The general structure is of the form

$$p(x_1,...,x_N)= \prod_{i=1}^N p(x_i|x_{\text{Pa}(i)})$$

In this specific example, we have
$$p(\{x\})=p(x_{A})p(x_{S})p(x_{T}|x_{A})p(x_{L}|x_{S})p(x_{B}|x_{S})p(x_{E}|x_{L},x_{T})p(x_{D}|x_{B},x_{E})p(x_{X}|x_{E})$$

#### Markov random field (MRF)

<center><img src="figs/mrf_yedidia.png" width=300></center>

The joint probability for a pairwise Markov Random Field (MRF) reads

$$p(\{x\},\{y\})=\frac{1}{Z}\prod_{\langle ij)}\psi_{ij}(x_{i},x_{j})\prod_{i}\phi_{i}(x_{i},y_{i})$$

Since we usually condition on the observations $y$, we will subsequently call the *evidence* terms simply by $\phi_{i}(x_{i})$.

## Equivalence of the three representations

Converting a MRF and a Bayesian network into a Factor Graph is trivial: each interaction among variables carry its own factor.

##### MRF to Factor Graph
<center><img src="figs/mrf_to_factor_yedidia.png" width=300></center>

##### Bayesian network to Factor Graph

<center><img src="figs/bayesian_network_to_factory_yedidia.png" width=300></center>

The inverse conversions can be obtained by
* inserting appropriate observation nodes
* introducing collective nodes for factors involving more than two variables nodes in the factor graph

##### Factor Graph to Bayesian network

<center><img src="figs/factor_to_bayesian_yedidia.png" width=300></center>

##### Factor Graph to MRF

<center><img src="figs/factor_to_mrf_yedidia.png" width=300></center>

Owing to this equivalence we will follow [Understanding belief propagation and its generalizations](https://www.cs.huji.ac.il/course/2005/pmai/tirguls/TR2001-22.pdf) and work with MRF, where BP updated are very simple and the introduction of the Bethe Free energy is simplified.

In particular, since we usually condition on the observations, we will drop the dependence on $y$ and subsequently call the *evidence* terms simply by $\phi_{i}(x_{i})$. We will thus write for the joint distribution of a MRF:

$$p(\{x\})=\frac{1}{Z}\prod_{\langle ij)}\psi_{ij}(x_{i},x_{j})\prod_{i}\phi_{i}(x_{i})$$

We will also use $b$ (which stands for *belief*) instead of $q$ for the variational distibution.

## From mean-field...

Recall that the variational approach consists in minimizing
$$\mathrm{KL}(b||p)$$
using a variational ansatz $b$. In the mean-field (MF) case, one employs a factorized variational distribution $b(x)= \prod_i b_i(x_i)$, thus obtaining:
$$\mathrm{KL}(b||p)=\sum_i \sum_{x_i} b_i(x_i) \log b_i(x_i)-\sum_x \prod_i q_i(x_i) \log p(x)$$

Writing the MRF distribution under the form of an exponential model (we set $\beta=1$ here for simplicity)
$$p(x)= \frac{1}{Z}e^{-E\left(x\right)}$$

we have an energy function
$$E\left(\left\{ x\right\} \right)=-\sum_{\left(i,j\right)}\log\psi_{ij}\left(x_{i},x_{j}\right)-\sum_{i}\log\phi_{i}\left(x_{i}\right)$$

from which the KL divergence is 
$$\mathrm{KL}(b||p)= G(\lbrace b_i\rbrace) + \log Z$$

where the mean-field free energy, expressed as a functional of the (normalized) beliefs $b_i$, reads

$$G_\text{MF} = U_\text{MF} - S_\text{MF}$$
with the average **energy** and **entropy** terms:
$$U_{\text{MF}}=-\sum_{\left(i,j\right)}\sum_{x_{i},x_{j}}b_{i}\left(x_{i}\right)b_{j}\left(x_{j}\right)\log\psi_{ij}\left(x_{i},x_{j}\right)-\sum_{i}\sum_{x_{i}}b_{i}\left(x_{i}\right)\log\phi_{i}\left(x_{i}\right)
$$
$$S_{\text{MF}}=-\sum_{i}\sum_{x_{i}}b_{i}\left(x_{i}\right)\log b_{i}\left(x_{i}\right)$$

## ...to Bethe free energy

Let us assume that the underlying structure of our MRF is that of a tree. In such a case, one can easily prove (by induction) that:

$$b\left(\left\{ x\right\} \right)=\prod_{\left(i,j\right)}b_{ij}\left(x_{i},x_{j}\right)\prod_{i}b_{i}\left(x_{i}\right)^{1-q_{i}}$$

with $b_{ij}\left(x_{i},x_{j}\right)$ the two-variable marginal and $q_i$ the degree (number of neighbors) of node $i$ in the graph.

Plugging this formula in the expression for the free energy we get the so-called **Bethe free energy**
$$G_{\text{Bethe}}=U_{\text{Bethe}}-S_{\text{Bethe}}$$
where the **average energy** is:
$$U_{\text{Bethe}}=-\sum_{\left(i,j\right)}\sum_{x_{i},x_{j}}b_{ij}\left(x_{i},x_{j}\right)\log\psi_{ij}\left(x_{i},x_{j}\right)-\sum_{i}\sum_{x_{i}}b_{i}\left(x_{i}\right)\log\phi_{i}\left(x_{i}\right)$$
and the **entropy** term is:
$$S_{\text{Bethe}}=-\sum_{\left(i,j\right)}\sum_{x_i,x_j} b_{ij}(x_i,x_j)\log b_{ij}(x_i,x_j)+\sum_i(q_i-1)\sum_{x_i} b_i(x_i)\log b_i(x_i)$$
It is instructive to think of the first term in the previous expression as as the entropy of all pair beliefs $b_{ij}$,  from which we subtract single variable entropies to compensate for overcounting.

We can also introduce the following definitions for the local energies
$$E_{i}\left(x_{i}\right)=-\log\phi_{i}\left(x_{i}\right)$$
$$E_{ij}\left(x_{i},x_{j}\right)=-\log\psi_{ij}\left(x_{i},x_{j}\right)-\log\phi_{i}\left(x_{i}\right)-\log\phi_{j}\left(x_{j}\right)$$
to write $U_{\text{Bethe}}$ as
$$U_{\text{Bethe}}=\sum_{\left(i,j\right)}\sum_{x_{i},x_{j}}b_{ij}\left(x_{i},x_{j}\right)\log\psi_{ij}\left(x_{i},x_{j}\right)-\sum_{i}\left(q_{i}-1\right)\sum_{x_{i}}b_{i}\left(x_{i}\right)\log\phi_{i}\left(x_{i}\right)$$


which displays the same structure of $S_{\text{Bethe}}$ [if you went through the *Bonus* derivation of the transfer matrix method for the 2d Ising model, you will probably experience a *deja-vu*, which is notoriously a sign of a glitch in the (transfer) matrix].

$G_\text{Bethe}$ is thus a functional of the two- and one-variable marginals, provided they are normalized
$$\sum_{x_i,x_j}b_{ij}(x_i,x_j)=1$$
$$\sum_{x_i}b_{i}(x_i)=1$$

and respect the local **marginal consistency** conditions
$$\sum_{x_j}b_{ij}(x_i,x_j)=b_{i}(x_i)$$


##### Note:
* In a tree, Bethe free energy is exact, therefore the beliefs $b$ are the true marginal;
* When the two-variable marginals are correct, $U_\text{Bethe}$ is exact for a MRF, since $\psi_{ij}$ only depend on the two variables $x_i$ and $x_j$, whereas the entropy term is an approximation: this will lead us to the introduction of more general free energies and corresponding generalized Belief Propagation methods;
* In a graph with loops the pairwise factorized $b$ is not a valid variational ansatz, therefore $G_\text{Bethe}$ is not generically an upper bound to the true free energy $-\log Z$ of the model $p$;

## Extremization of the Bethe free energy

We can impose the constraints of local marginal consistency through a set of Lagrange multipliers $\lambda_{ij}(x_j)$, $\gamma_{ij}$, and $\gamma_i$. We thus have a Lagrangian $\mathcal{L}$:
$$
\mathcal{L}=G_{\text{Bethe}}+\sum_{(i,j),(j,i)}\sum_{x_j}\lambda_{ij}(x_j)\left(\sum_{x_i}b_{ij}(x_i,x_j)-b_j(x_j)\right)+
\sum_{(i,j)}\gamma_{ij}\left(\sum_{x_i,x_j}b_{ij}(x_i,x_j)-1\right)+\sum_{i}\gamma_{i}\left(\sum_{x_i}b_{i}(x_i)-1\right)
$$

Differentiating with respect to $b_{ij}(x_i,x_j)$ and $b_{i}(x_i)$ and setting the derivatives to zero yields the relationships for the beliefs:
$$
b_{ij}(x_i,x_j)\propto \psi_{ij}(x_i,x_j) \exp\left(\lambda_{ij}(x_j)+\lambda_{ji}(x_i)\right)$$
$$b_i(x_i)\propto \phi_i(x_i)^{\frac{1}{1-q_i}}\exp\left(\frac{1}{q_i-1}\sum_{j\in N(i)} \lambda_{ji}(x_i)\right)$$

Using an insane amount of intuition, we define the **messages** $m$ by the following expression:
$$\lambda_{ij}\left(x_{j}\right)=\log\phi_j(x_j) + \log\prod_{k\in N\left(j\right)\setminus i}m_{kj}\left(x_{j}\right)$$

Pluggin this expression in the equations for the $b$ one gets:

$$b_{i}\left(x_{i}\right)\propto\phi_{i}\left(x_{i}\right)\prod_{k\in N\left(i\right)}m_{ki}\left(x_{i}\right)$$
$$b_{ij}\left(x_{i},x_{j}\right)\propto\psi_{ij}\left(x_{i},x_{i}\right)\phi_{i}\left(x_{i}\right)\phi_{j}\left(x_{j}\right)\prod_{k\in N\left(i\right)\setminus j}m_{ki}\left(x_{i}\right)\prod_{k\in N\left(j\right)\setminus i}m_{lj}\left(x_{j}\right)$$

The final step is to enforce the **marginal consistency constraint** $b_j(x_j)=\sum_{x_i}b_{ij}(x_i,x_j)$, which leads directly to the so-called BP equations:

$$m_{ij}(x_j)\propto \sum_{x_i} \psi_{ij}(x_i,x_j)\phi_i(x_i)\prod_{k\in N(i)\setminus j} m_{ki}(x_i)$$

##### Note:

* When the graph is a **tree**, the Bethe free energy $F(b)$ is convex;
* In general (on loopy graphs), $F(b)$ is **not convex**, and the convergence of Loopy BP is **not guaranteed**.

## Belief Propagation

The stationary points of $G_\text{Bethe}$ are in one-to-one correspondence with the fixed point of the **Belief Propagation** algorithm.

The BP equations can be considered as describing **message-passing** on a graph.

For a **tree**, we initialize the messages at all the leaves and pass messages throughout the tree. The message from node $i$ to node $j$ is given by:

$$m_{i j} (x_j) \propto \sum_{x_i} \psi(x_i,x_j) \prod_{k\in N(i)\setminus j} m_{k i}(x_i)$$

where $N(i)$ is the set of neighbors of node $i$ in the graph. The algorithm terminates once every vertex has sent messages in both directions:

<center><img src="figs/message_passing_mrf_yedidia.png" width=300></center>

At convergence, the marginal probability is recovered through:
$$p(x_i)\propto \prod_{k\in N(i)} m_{k i}(x_i)$$

In a general graph with loops, one initializes **all messages** and iterates until convergence (which is not guaranteed). The marginals can be thus estimated through the one- and two-variable beliefs:

$$b_i(x_i) \propto \phi_i(x_i)\prod_{k\in N(i)} m_{k i}(x_i)$$
$$b_{ij}(x_i,x_j)\propto \psi_{ij}(x_i,x_j)\phi_i(x_i)\phi_j(x_j)\prod_{k\in N(i)\setminus j} m_{k i}(x_i) \prod_{l\in N(j)\setminus i} m_{l j}(x_j)$$

## Back to Factor Fraphs: the sum-product algorithm

Let us go back to the factor graph representation of the probability distribution $p$. In such a case, one introduced two types of (*cavity*) messages:
* variable-to-factor messages $\mu_{j\to f}$
* factor-to-variable messages $\mu_{f\to j}$

The update equations take the form:

$$\mu_{j\to f}(x_j)\propto \prod_{h\in N(j) \setminus f} \mu_{h\to j}(x_j)\qquad$$
$$\mu_{f\to i}(x_i) \propto \sum_{\lbrace x \rbrace_f\setminus i} \psi_f(\lbrace x \rbrace_f)\prod_{j\in N(f)\setminus i} \mu_{j\to f}(x_j)$$
$N_i$ is the set of factors that contain variable $i$:

<center><img src="figs/message_passing_factor_graph.png" width=400></center>

##### Note:
A similar Bethe free energy can be introduced in terms of the (cavity) messages $\mu$. By virtue of the equivalence between MRF and Factor graphs, the fixed point of BP in the $\mu$ are in one-to-one correspondence of the Bethe free energy.

The sum-product update equations can be summarized in the following Algorithm:

**The message sent from a node $v$ on an edge $e$ is the product of the local function at $v$ (or the unit function if $v$ is a variable node) with all messages received at $v$ on edges other than $e$, summarized for the variable associated with $e$.**

Noting that incoming messages onto a variable node are all functions of the same variable, the marginalization (summarization) is vacuous, and associating an identity function to each variable node, we arrive at the previous update equations.

In [Factor graphs and the sum-product algorithm](https://ieeexplore.ieee.org/document/910572) the authors note that this is implied by the fact the an acyclic factor graph is both a representation of the independence/factorization property of $p$ and at the same time of the *expression tree*/*computational graph* of its marginal functions.

Consider again the previous example of a factor graph:

<center><img src="figs/factor_graph_kschischang.png" width=300></center>

Computing the marginal $p(x_1)$ can be visualized by reorganizing the graph as a rooted tree:

<center><img src="figs/single_marginal_rooted_tree.png" width=300></center>

and considering the following expression tree:

<center><img src="figs/single_marginal_expression_tree.png" width=300></center>

## Max product

The max operation obey the same distribution rule as the sum. For this reason, one can find the configurations with maximal probability $p(x)=\frac{1}{Z}\prod_{(ij)}\psi_{ij}(x_i,x_j)\prod_i\phi_i(x_i)$ for a tree-structured factor graph using the so-called **max-product** update equations:

$$m_{i j}(x_j)\propto \max_{x_i} \psi_{ij}(x_i,x_j)\phi_i(x_i)\prod_{k\in N(i) \setminus j}m_{k i}(x_i)\qquad b_i(x_i)\propto \phi_i(x_i) \prod_{j\in N(i)} m_{ji}(x_i)$$

The max-product solution is $x^*=(x_1^*,\ldots, x_n^*)$ with $x_i^*=\text{argmax}_{x_i} b_i(x_i)$.

While max-product algorithm is exact when the graph is a tree, for graphs with loops max-product it has somewhat worse convergence than sum-product (see [Fixing Max-Product: Convergent Message Passing Algorithms for MAP LP-Relaxations](https://papers.nips.cc/paper_files/paper/2007/hash/8d6dc35e506fc23349dd10ee68dabb64-Abstract.html))

It is sometimes convenient to write a logarithmic version of the max product algorithm with $\psi_{ij}(x_i,x_j)=e^{E_{ij}(x_i,x_j)}, \phi_i(x_i)=e^{E_i(x_i)}$ and $m_{ij}(x_j)=e^{\mu_{ij}(x_j)}$:
$$\mu_{i j}(x_j)\propto \max_{x_i} \left(E_{ij}(x_i,x_j)+E_i(x_i)+\sum_{k\in N(i) \setminus j}\mu_{k i}(x_i)\right)$$
This version is sometimes referred to as the **max-sum** algorithm.

## Application #1

#### BP for a binary variable model

Consider a model with binary variables
$$p(x)=\frac{1}{Z} \exp{\left(\sum_{(ij)}J_{ij}x_ix_j +\sum_i h_i x_i\right)}$$

Interactions are pairwise so we can use a single set of messages, just like in the MRF case.

The BP equations are:
$$m^{t+1}_{ij}(x_j)\propto \sum_{x_i}e^{J_{ij}x_ix_j +h_i x_i} \prod_{k\in N(i) \setminus j}m^t_{k i}(x_i)$$

The messages $m_{ij}(x_j)$ are distributions over a single bit $x_j$ and are specified by a single number. Convenient parametrizations are either $\mu_{ij}$ or $a_{ij}$:
$$m_{ij}(x_j)=\frac{1}{2}(1+\mu_{ij}x_j) =\frac{e^{a_{ij}x_j}}{2\cosh(a_{ij})}$$
with  $\mu_{ij}=\tanh(a_{ij})$.

In terms of $a_{ij}$ the BP equations become

$$m^{t+1}_{ij}(x_j)\propto \sum_{x_i} \exp\left(J_{ij}x_ix_j+h_i x_i+\sum_{k\in N(i)\setminus j} a^t_{ki}x_i\right)\propto 2 \cosh \left(J_{ij} x_j+ h_i +\sum_{k\in N(i)\setminus j} a^t_{ki}\right)$$

You will prove that an equivalent form is the following:

$$\mu^{t+1}_{ij} = \frac{m^{t+1}_{ij}(x_j=1)-m^{t+1}_{ij}(x_j=-1)}{m^{t+1}_{ij}(x_j=1)+m^{t+1}_{ij}(x_j=-1)}=\tanh\left(h_i + \sum_{k\in N(i)\setminus j}a^t_{ki}\right)\tanh(J_{ij})$$

#### *Bonus: Convergence of BP on a binary factor graph with pairwise interactions*

We can use the last expression to figure out the convergence property of BP on a graph [assuming the algorithm is run in parallel].

Since $\mu=\tanh (a)$, we have a map
$$a_{ij}^{t+1}=f_{ij}(a^t)$$

Let us work out its Jacobian:

$$\frac{\partial a_{ij}^{t+1}}{\partial a_{i'j'}^t}=\frac{d a^{t+1}_{ij}}{d\mu^{t+1}_{ij}}\frac{\partial \mu^{t+1}_{ij}}{\partial a^{t}_{i'j'}}$$
$$=\underbrace{\frac{1-\tanh^2\left(h_i +\sum_{k\in N(i)\setminus j} a^t_{ki}\right)}{1-\mu^{t+1}_{ij}(a^t)}\text{sign}(J_{ij})}_{B_{ij}(a^t)}\underbrace{\tanh(|J_{ij}|)\delta_{j'i}\left(1-\delta_{i'j}\right)}_{A_{ij,i'j'}}$$
It is easy to show that $|B_{ij}(a^t)|\le 1$. Thus, a sufficient condition for the convergence of parallel BP is that the spectral radius of $A$ is less than 1 ($\rho(A)<1$) [This condition is independent of $h_i$].

Find more details for sum-product on general factor graphs in [this paper](https://ieeexplore.ieee.org/document/4385778).

## Application #2

#### Clustering

Consider the problem of clustering the data points $x_i, i=1,\ldots,N$ with similarities $s(i,j)=-|x_i-x_j|^2$, with centers defined by exemplars.

The paper [Clustering by Passing Messages Between Data Points](https://www.science.org/doi/10.1126/science.1136800) addresses the problem using a novel method that they call **affinity propagation**, which can be formulated as **max-sum** on an appropriate factor graph.

Assign each data point $i$ to a cluster $c_i\in{1,\ldots,N}$ [if $c_i=i$, the data point $i$ is a cluster center].

The objective is to maximize
$$E(c_1,\ldots,c_N)=\sum_{i=1}^N s(i,c_i) +\sum_{i<k }^N\delta_{k} (c_i,c_k)$$
$$\delta_{k}(c_i,c_k)=-\infty\qquad \text{if}~ c_i=k~ \text{and} ~c_k\ne k$$
$$= 0 \quad\text{otherwise}$$
The first term aims to find a center $c_i$ near $i$. The second term ensures that point $i$ cannot choose $k$ as its center ($c_i=k$) if $k$ is not
a center ($c_k\ne k$).

We can use **max-sum** to solve the problem on the corresponding factor graph:

<center><img src="figs/affinity_factor_graph.png" width=500></center>

The corresponding **affinity propagation** equations can be written in terms of the classic **responsibilities** and the so-called **availabilities** and take the following form:

$$r\left(i,k\right)=s\left(i,k\right)-\max_{l\neq k}\left[a\left(i,l\right)+s\left(i,l\right)\right]$$
$$a\left(i,k\right)=\min\left\{ 0,r\left(k,k\right)+\sum_{j\notin\left\{ i,k\right\} }\max\left[0,r\left(j,k\right)\right]\right\}$$

The relevant marginals are represented by the *self-availabilities*

$$a\left(k,k\right)=\sum_{j\neq k}\max\left[0,r\left(j,k\right)\right]$$

Here is a pictorial representation of the update equations:

<center><img src="figs/affinity_availabilities.png" width=500></center>


Here's an example of the message passing dynamics for a set of points in two dimensions:

<center><img src="figs/affinity_propagation_frey.png" width=600></center>

## Application #3

#### Epidemic mitigation

You may want to look at these works on COVID-19 mitigation strategies using BP-based inference on contact tracing data

* [Inference of causality in epidemics on temporal contact networks](https://www.nature.com/articles/srep27538)
* [Epidemic mitigation by statistical inference from contact tracing data](https://www.pnas.org/doi/abs/10.1073/pnas.2106548118)

by yours truly.

# <center>Assignments</center>

#### Ex 5.1

In this exercise, you will use BP to compute marginals in a model with binary variables (the Ising model) defined on graphs with various level of sparsity. Before implementing BP, make sure to prove the relevant relations involving one- and two-variable marginals detailed in the last cell.

#### Setting

The Ising model (here written at temperature $1$) is a probability distribution over vectors of binary variables $x=(x_1,\ldots,x_N)$ with $x_i=\pm 1$:

$$p(x)=\frac{1}{Z}\exp \left(-E(x)\right)\qquad E(x)=-\frac{1}{2}\sum_{i\ne j}J_{ij}x_ix_j-\sum_i h_i x_i$$

$$Z=\sum_x \exp(-E(x))=\sum_{x_1}\ldots \sum_{x_N} \exp(-E(x))$$

where in general $J_{ij}=J_{ji}$ and $J\neq 0$ when $i$ and $j$ are neighbours on a given (undirected) graph, and $0$ otherwise. We will start considering Ising models defined on simple lattices.

#### Exercise

For each graph, compute both the single-variable magnetizations $m_{i}^\text{exact}=\langle x_{i}\rangle$ and the spin correlations $\rho_{ij}^\text{exact}=\left\langle x_{i}x_{j}\right\rangle -\left\langle x_{i}\right\rangle \left\langle x_{j}\right\rangle $ exactly using explicit enumeration. Compare the magnetization and the correlations obtained through BP with the exact values.

##### Fully Connected Ising Model

Consider an Ising model on a complete (fully connected) graph. The weights $J_{ik}$ are symmetric and random, each drawn from a Gaussian distribution with variance $\frac{J^{2}}{N}$, where $N=20$ is the number of spins. The $h_{i}$ fields are i.i.d. normally distributed.

Set $J=\beta$. Use BP to compute both mean values of each $x_i$ and the correlations among spins. Plot the errors in estimating these quantities as a function of $\beta$ for $0<\beta\le2$ [Estimate the mean and standard deviation of the errors from multiple random instances for each $\beta$].

##### Sparse Ising Model

Consider an Ising model defined on a sparse graph, with $c$ being the approximate fraction of edges wrt $N$ (you can construct is using a probability for each edge equal to $\frac{c}{N}$, but make sure that $J_{ij}=J_{ji}$). Take each $J_{ij}$ to be either $+\beta$ of $-\beta$ at random. Set $h_i=\theta > 0$. You can check connectedness using [networkx](https://networkx.org/en/).

Plot the mean and standard deviation of the estimation error as a function of $c$ (for various $\beta$) and $\beta$ (for various $c$) for $0<c\le1$ and $0<\beta\le1$ and for various $\theta$ [avoid using 2d or 3d plots, it is better to plot multiple curves on the same plot].

#### Some details on marginals in BP

Prove the last relation in the section **Application #1**. In your implementation, you may want to use the $a_{ij}$ parametrization:

$$a_{ij}=\tanh^{-1}(\mu_{ij})=\tanh^{-1}\left(\frac{m_{ij}(x_{j}=1)-m_{ij}(x_{j}=-1)}{m_{ij}(x_{j}=1)+m_{ij}(x_{j}=-1)}\right)=\frac{1}{2}\log\frac{m_{ij}(x_{j}=1)}{m_{ij}(x_{j}=-1)}$$

For the joint probability for two spins $x_i$ and $x_j$, we have (prove it):
$$b_{ij}(x_{i},x_{j})\propto\exp\left(J_{ij}x_{i}x_{j}+h_{i}x_{i}+h_{j}x_{j}+\sum_{k\in N(i)\backslash j}a_{ki}x_{i}+\sum_{l\in N(j)\backslash i}a_{lj}x_{j}\right)$$